In [1]:
import scanpy as sc
import anndata as ad
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.colors import ListedColormap, rgb2hex
import numpy as np
import warnings
import pandas as pd
warnings.filterwarnings('ignore')
import numpy as np
from sklearn.metrics import jaccard_score
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['pdf.fonttype'] = 42 # ADOBE AI 字帖

from matplotlib.font_manager import fontManager, FontProperties

fontManager.addfont('/data/work/Arial.ttf')

font = FontProperties(fname='/data/work/Arial.ttf')
font_name = font.get_name()
plt.rcParams['font.family'] = font_name

In [2]:
adata = sc.read_h5ad('/data/work/05.cluster/FuseMap/0106/Hippocampus_latent_embeddings_all_single_pretrain/dmt_leiden_20250108_1.h5ad')
names = [
    'B03607C4E6_WT2024071214941.h5ad',
    
    '12_B03605F3G5_WT202403310048.h5ad',
    '13_B03612A1C3_WT202403310056.h5ad',
    '14_A03591A1C3_WT202403310045.h5ad',
    '16_A03592A4C6_WT202403310044.h5ad',
    '18_B03602C4D6_WT202405020031.h5ad',
    '20_B03606F3G5_WT202405020032.h5ad',
    '22_B03606C4E6_WT202403310050.h5ad',
    '23_B03609A4D6_WT202404150263.h5ad',
    '27_B03610C1E3_WT202403310051.h5ad',
    '31_B03619A1D3_WT202403310052.h5ad',
    '35_B03619E4G6_WT202403310053.h5ad',
    '39_A03589A1D4_WT202403310046.h5ad',
    '43_A03590E1G4_WT202403310064.h5ad',
    '47_A03593C1F3_WT202403310068.h5ad',
]

In [3]:
adata.obs_names_make_unique()
dic_dmt_leiden = {
    '0': 'hip_sc_01',
    '7': 'hip_sc_02',
    
    '1': 'hip_sc_03',
    '3': 'hip_sc_04',
    '6': 'hip_sc_04',
    '14': 'hip_sc_05',
    
    
    '2': 'hip_sc_06',
    '13': 'hip_sc_07',
    '19': 'hip_sc_08',
    '20': 'hip_sc_09',
    
    '4': 'hip_sc_10',
    '21': 'hip_sc_10',
    
    '8': 'hip_sc_11',
    '10': 'hip_sc_12',
    '22': 'hip_sc_13',
    
    '9': 'hip_sc_14',
    '11': 'hip_sc_14',
    '16': 'hip_sc_15',
    '18': 'hip_sc_15',
    '26': 'hip_sc_15',
    '31': 'hip_sc_15',
    
    '12': 'hip_sc_16',
    '15': 'hip_sc_17',
    '25': 'hip_sc_17',
    '29': 'hip_sc_17',
    '17': 'hip_sc_18',
    
    '24': 'z_delete',
    
    
    '5': 'hip_sc_19',
    '23': 'hip_sc_20',
    '27': 'hip_sc_21',
    '28': 'hip_sc_22',
    '30': 'hip_sc_23',
}
adata.obs['dmt_leiden_anno'] = [dic_dmt_leiden[i] for i in adata.obs['dmt_leiden']]


In [5]:
colormap = {
  'hip_sc_01' : '#9b38e9',
  'hip_sc_02' : '#a89630',
  'hip_sc_03': '#5b798b',
  'hip_sc_04' : '#cb2505',
  'hip_sc_05' : '#62e7dd',
  'hip_sc_06' : '#245200',
  'hip_sc_07' : '#374898',
  'hip_sc_08' : '#6d85c7',
  'hip_sc_09' : '#35c498',
  'hip_sc_10' : '#9e2dc6',
  'hip_sc_11' : '#2d7476',
  'hip_sc_12' : '#cb0d6c',
  'hip_sc_13' : '#20ea38',
  'hip_sc_14' : '#0fabb6',
  'hip_sc_15' : '#a59099',
  'hip_sc_16' : '#2bea3a',
  'hip_sc_17' : '#17b064',
  'hip_sc_18' : '#52b8d5',
  'hip_sc_19' : '#da2ef2',
  'hip_sc_20' : '#6240f7',
  'hip_sc_21' : '#c47233',
  'hip_sc_22':'#a83b23',
  'hip_sc_23':'#9994da'
}

In [7]:
adata = adata[adata.obs['dmt_leiden_anno'] != 'z_delete']

In [8]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

# 创建画布和网格布局
fig = plt.figure(figsize=(64, 6))
gs = GridSpec(1, 15, figure=fig)

# 计算全局坐标范围
x_min, x_max, y_min, y_max = float('inf'), float('-inf'), float('inf'), float('-inf')
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    
    x_min = min(x_min, adata_temp.obsm['align_spatial_2d'][:, 0].min())
    x_max = max(x_max, adata_temp.obsm['align_spatial_2d'][:, 0].max())
    y_min = min(y_min, adata_temp.obsm['align_spatial_2d'][:, 1].min())
    y_max = max(y_max, adata_temp.obsm['align_spatial_2d'][:, 1].max())

# 绘制子图
count = 0
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    col = count % 15
    ax = fig.add_subplot(gs[0, col])
    
    # 绘制嵌入图
    sc.pl.embedding(
        adata_temp, basis="align_spatial_2d", color='dmt_leiden_anno',
        show=False, s=1, title='', legend_loc=None, ax=ax, palette=colormap, groups = ['hip_sc_01', 'hip_sc_02']
    )
    
    # 设置统一的坐标轴范围
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    
    # 关闭坐标轴并设置等比例缩放
    ax.axis('off')
    ax.set_aspect('equal')
    # scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
    # ax.add_artist(scalebar)
    if count == 0:
        scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
        ax.add_artist(scalebar)
    count += 1
plt.savefig(f'/data/work/05.cluster/FuseMap/0116/hip_sc_0102/0102.png', bbox_inches = 'tight', dpi = 600)
plt.close()
# plt.show()

In [9]:
adata = adata[adata.obs['dmt_leiden_anno'].isin(['hip_sc_01', 'hip_sc_02'])].copy()

In [10]:
adatas = []
for i in set(adata.obs['slice_code']):
    i_ = i.replace('.h5ad', '')
    temp = adata[adata.obs['slice_code'] == i].copy()
    sc.pp.normalize_total(temp)
    sc.pp.log1p(temp)
    sc.pp.scale(temp, zero_center=False, max_value=10)
    sc.tl.rank_genes_groups(temp, 
                        groupby = 'dmt_leiden_anno', 
                        method = 'wilcoxon', 
                        pts = True)
    
    degs = []
    for i in ['hip_sc_01', 'hip_sc_02']:
        t = sc.get.rank_genes_groups_df(temp, i)
        t = t[t['logfoldchanges'] > 0.5]
        t = t[t['pvals'] < 0.01]
        t = t[t['pvals_adj'] < 0.01]
        t['dmt_leiden_anno'] = i
        degs.append(t)
    deg = pd.concat(degs)
    deg.to_csv(f'/data/work/05.cluster/FuseMap/0116/hip_sc_0102/{i_}.csv')
    if len(i.split('_')) == 3:
        adatas.append(temp)
adata = ad.concat(adatas)

In [12]:
sc.tl.rank_genes_groups(adata, 
                        groupby = 'dmt_leiden_anno', 
                        method = 'wilcoxon', 
                        pts = True)

In [13]:
degs = []
for i in ['hip_sc_01', 'hip_sc_02']:
    t = sc.get.rank_genes_groups_df(adata, i)
    t = t[t['logfoldchanges'] > 0.5]
    t = t[t['pvals'] < 0.01]
    t = t[t['pvals_adj'] < 0.01]
    t['dmt_leiden_anno'] = i
    degs.append(t)
deg = pd.concat(degs)
deg.to_csv(f'/data/work/05.cluster/FuseMap/0116/hip_sc_0102/gw13.csv')

In [7]:

adatas = []
for i in set(adata.obs['slice_code']):
    temp = adata[adata.obs['slice_code'] == i].copy()
    sc.pp.normalize_total(temp)
    sc.pp.log1p(temp)
    sc.pp.scale(temp, zero_center=False, max_value=10)
    adatas.append(temp)
adata = ad.concat(adatas)

In [8]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

fig = plt.figure(figsize=(64, 6))
gs = GridSpec(1, 15, figure=fig)

# 计算全局坐标范围
x_min, x_max, y_min, y_max = float('inf'), float('-inf'), float('inf'), float('-inf')
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    
    x_min = min(x_min, adata_temp.obsm['align_spatial_2d'][:, 0].min())
    x_max = max(x_max, adata_temp.obsm['align_spatial_2d'][:, 0].max())
    y_min = min(y_min, adata_temp.obsm['align_spatial_2d'][:, 1].min())
    y_max = max(y_max, adata_temp.obsm['align_spatial_2d'][:, 1].max())

# 绘制子图
count = 0
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    col = count % 15
    ax = fig.add_subplot(gs[0, col])
    
    # 绘制嵌入图
    sc.pl.embedding(
        adata_temp, basis="align_spatial_2d", color='FOSL2',
        show=False, s=1, title='', legend_loc=None, ax=ax, cmap = 'Reds'    )
    
    # 设置统一的坐标轴范围
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    
    # 关闭坐标轴并设置等比例缩放
    ax.axis('off')
    ax.set_aspect('equal')
    # scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
    # ax.add_artist(scalebar)
    if count == 0:
        scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
        ax.add_artist(scalebar)
    count += 1
plt.savefig(f'/data/work/05.cluster/FuseMap/0116/hip_sc_0102/FOSL2.png', bbox_inches = 'tight', dpi = 600)
plt.close()
# plt.show()

In [9]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

fig = plt.figure(figsize=(64, 6))
gs = GridSpec(1, 15, figure=fig)

# 计算全局坐标范围
x_min, x_max, y_min, y_max = float('inf'), float('-inf'), float('inf'), float('-inf')
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    
    x_min = min(x_min, adata_temp.obsm['align_spatial_2d'][:, 0].min())
    x_max = max(x_max, adata_temp.obsm['align_spatial_2d'][:, 0].max())
    y_min = min(y_min, adata_temp.obsm['align_spatial_2d'][:, 1].min())
    y_max = max(y_max, adata_temp.obsm['align_spatial_2d'][:, 1].max())

# 绘制子图
count = 0
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    col = count % 15
    ax = fig.add_subplot(gs[0, col])
    
    # 绘制嵌入图
    sc.pl.embedding(
        adata_temp, basis="align_spatial_2d", color='SIAH3',
        show=False, s=1, title='', legend_loc=None, ax=ax, cmap = 'Reds'    )
    
    # 设置统一的坐标轴范围
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    
    # 关闭坐标轴并设置等比例缩放
    ax.axis('off')
    ax.set_aspect('equal')
    # scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
    # ax.add_artist(scalebar)
    if count == 0:
        scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
        ax.add_artist(scalebar)
    count += 1
plt.savefig(f'/data/work/05.cluster/FuseMap/0116/hip_sc_0102/SIAH3.png', bbox_inches = 'tight', dpi = 600)
plt.close()
# plt.show()

In [10]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

fig = plt.figure(figsize=(64, 6))
gs = GridSpec(1, 15, figure=fig)

# 计算全局坐标范围
x_min, x_max, y_min, y_max = float('inf'), float('-inf'), float('inf'), float('-inf')
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    
    x_min = min(x_min, adata_temp.obsm['align_spatial_2d'][:, 0].min())
    x_max = max(x_max, adata_temp.obsm['align_spatial_2d'][:, 0].max())
    y_min = min(y_min, adata_temp.obsm['align_spatial_2d'][:, 1].min())
    y_max = max(y_max, adata_temp.obsm['align_spatial_2d'][:, 1].max())

# 绘制子图
count = 0
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    col = count % 15
    ax = fig.add_subplot(gs[0, col])
    
    # 绘制嵌入图
    sc.pl.embedding(
        adata_temp, basis="align_spatial_2d", color='NNAT',
        show=False, s=1, title='', legend_loc=None, ax=ax, cmap = 'Reds'    )
    
    # 设置统一的坐标轴范围
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    
    # 关闭坐标轴并设置等比例缩放
    ax.axis('off')
    ax.set_aspect('equal')
    # scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
    # ax.add_artist(scalebar)
    if count == 0:
        scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
        ax.add_artist(scalebar)
    count += 1
plt.savefig(f'/data/work/05.cluster/FuseMap/0116/hip_sc_0102/NNAT.png', bbox_inches = 'tight', dpi = 600)
plt.close()
# plt.show()

In [11]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

fig = plt.figure(figsize=(64, 6))
gs = GridSpec(1, 15, figure=fig)

# 计算全局坐标范围
x_min, x_max, y_min, y_max = float('inf'), float('-inf'), float('inf'), float('-inf')
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    
    x_min = min(x_min, adata_temp.obsm['align_spatial_2d'][:, 0].min())
    x_max = max(x_max, adata_temp.obsm['align_spatial_2d'][:, 0].max())
    y_min = min(y_min, adata_temp.obsm['align_spatial_2d'][:, 1].min())
    y_max = max(y_max, adata_temp.obsm['align_spatial_2d'][:, 1].max())

# 绘制子图
count = 0
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    col = count % 15
    ax = fig.add_subplot(gs[0, col])
    
    # 绘制嵌入图
    sc.pl.embedding(
        adata_temp, basis="align_spatial_2d", color='NR4A2',
        show=False, s=1, title='', legend_loc=None, ax=ax, cmap = 'Reds'    )
    
    # 设置统一的坐标轴范围
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    
    # 关闭坐标轴并设置等比例缩放
    ax.axis('off')
    ax.set_aspect('equal')
    # scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
    # ax.add_artist(scalebar)
    if count == 0:
        scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
        ax.add_artist(scalebar)
    count += 1
plt.savefig(f'/data/work/05.cluster/FuseMap/0116/hip_sc_0102/NR4A2.png', bbox_inches = 'tight', dpi = 600)
plt.close()
# plt.show()

In [12]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

fig = plt.figure(figsize=(64, 6))
gs = GridSpec(1, 15, figure=fig)

# 计算全局坐标范围
x_min, x_max, y_min, y_max = float('inf'), float('-inf'), float('inf'), float('-inf')
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    
    x_min = min(x_min, adata_temp.obsm['align_spatial_2d'][:, 0].min())
    x_max = max(x_max, adata_temp.obsm['align_spatial_2d'][:, 0].max())
    y_min = min(y_min, adata_temp.obsm['align_spatial_2d'][:, 1].min())
    y_max = max(y_max, adata_temp.obsm['align_spatial_2d'][:, 1].max())

# 绘制子图
count = 0
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    col = count % 15
    ax = fig.add_subplot(gs[0, col])
    
    # 绘制嵌入图
    sc.pl.embedding(
        adata_temp, basis="align_spatial_2d", color='NRP1',
        show=False, s=1, title='', legend_loc=None, ax=ax, cmap = 'Reds'    )
    
    # 设置统一的坐标轴范围
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    
    # 关闭坐标轴并设置等比例缩放
    ax.axis('off')
    ax.set_aspect('equal')
    # scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
    # ax.add_artist(scalebar)
    if count == 0:
        scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
        ax.add_artist(scalebar)
    count += 1
plt.savefig(f'/data/work/05.cluster/FuseMap/0116/hip_sc_0102/NRP1.png', bbox_inches = 'tight', dpi = 600)
plt.close()
# plt.show()

In [13]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

fig = plt.figure(figsize=(64, 6))
gs = GridSpec(1, 15, figure=fig)

# 计算全局坐标范围
x_min, x_max, y_min, y_max = float('inf'), float('-inf'), float('inf'), float('-inf')
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    
    x_min = min(x_min, adata_temp.obsm['align_spatial_2d'][:, 0].min())
    x_max = max(x_max, adata_temp.obsm['align_spatial_2d'][:, 0].max())
    y_min = min(y_min, adata_temp.obsm['align_spatial_2d'][:, 1].min())
    y_max = max(y_max, adata_temp.obsm['align_spatial_2d'][:, 1].max())

# 绘制子图
count = 0
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    col = count % 15
    ax = fig.add_subplot(gs[0, col])
    
    # 绘制嵌入图
    sc.pl.embedding(
        adata_temp, basis="align_spatial_2d", color='SEMA3C',
        show=False, s=1, title='', legend_loc=None, ax=ax, cmap = 'Reds'    )
    
    # 设置统一的坐标轴范围
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    
    # 关闭坐标轴并设置等比例缩放
    ax.axis('off')
    ax.set_aspect('equal')
    # scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
    # ax.add_artist(scalebar)
    if count == 0:
        scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
        ax.add_artist(scalebar)
    count += 1
plt.savefig(f'/data/work/05.cluster/FuseMap/0116/hip_sc_0102/SEMA3C.png', bbox_inches = 'tight', dpi = 600)
plt.close()
# plt.show()

In [14]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

fig = plt.figure(figsize=(64, 6))
gs = GridSpec(1, 15, figure=fig)

# 计算全局坐标范围
x_min, x_max, y_min, y_max = float('inf'), float('-inf'), float('inf'), float('-inf')
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    
    x_min = min(x_min, adata_temp.obsm['align_spatial_2d'][:, 0].min())
    x_max = max(x_max, adata_temp.obsm['align_spatial_2d'][:, 0].max())
    y_min = min(y_min, adata_temp.obsm['align_spatial_2d'][:, 1].min())
    y_max = max(y_max, adata_temp.obsm['align_spatial_2d'][:, 1].max())

# 绘制子图
count = 0
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    col = count % 15
    ax = fig.add_subplot(gs[0, col])
    
    # 绘制嵌入图
    sc.pl.embedding(
        adata_temp, basis="align_spatial_2d", color='SEMA3A',
        show=False, s=1, title='', legend_loc=None, ax=ax, cmap = 'Reds'    )
    
    # 设置统一的坐标轴范围
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    
    # 关闭坐标轴并设置等比例缩放
    ax.axis('off')
    ax.set_aspect('equal')
    # scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
    # ax.add_artist(scalebar)
    if count == 0:
        scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
        ax.add_artist(scalebar)
    count += 1
plt.savefig(f'/data/work/05.cluster/FuseMap/0116/hip_sc_0102/SEMA3A.png', bbox_inches = 'tight', dpi = 600)
plt.close()
# plt.show()

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

fig = plt.figure(figsize=(64, 6))
gs = GridSpec(1, 15, figure=fig)

# 计算全局坐标范围
x_min, x_max, y_min, y_max = float('inf'), float('-inf'), float('inf'), float('-inf')
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    
    x_min = min(x_min, adata_temp.obsm['align_spatial_2d'][:, 0].min())
    x_max = max(x_max, adata_temp.obsm['align_spatial_2d'][:, 0].max())
    y_min = min(y_min, adata_temp.obsm['align_spatial_2d'][:, 1].min())
    y_max = max(y_max, adata_temp.obsm['align_spatial_2d'][:, 1].max())

# 绘制子图
count = 0
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    col = count % 15
    ax = fig.add_subplot(gs[0, col])
    
    # 绘制嵌入图
    sc.pl.embedding(
        adata_temp, basis="align_spatial_2d", color='NTSR1',
        show=False, s=1, title='', legend_loc=None, ax=ax, cmap = 'Reds'    )
    
    # 设置统一的坐标轴范围
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    
    # 关闭坐标轴并设置等比例缩放
    ax.axis('off')
    ax.set_aspect('equal')
    # scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
    # ax.add_artist(scalebar)
    if count == 0:
        scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
        ax.add_artist(scalebar)
    count += 1
plt.savefig(f'/data/work/05.cluster/FuseMap/0116/hip_sc_0102/NTSR1.png', bbox_inches = 'tight', dpi = 600)
plt.close()
# plt.show()